# Wikipedia community detection

Let's try to run the community detection via Infomap and Neuromap on a network as huge as the Greek, German, and English Wikipedia!

In [1]:
import sys
sys.path.append("..") # go to parent folder

import matplotlib.pyplot as plt
import numpy as np
import re
import networkx as nx
import igraph as ig
import infomap
import torch
from torch_geometric.nn   import GCN, GraphSAGE, GIN, GAT
from torch_geometric.data import Data
from typing import Tuple, Dict, List, Union


import src.neuromap as nm
import src.optimize as opt
import src.map_equation as meq
from src.utils import compare_partitions

In [2]:
# Prefer CUDA if available, else MPS (Apple Silicon), else CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print("Device:", device)

Device: cpu


In [3]:
def load_edgelist(filepath):
    edges = []
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("%") or line.strip() == "":
                continue
            
            try:
                src, dst = map(int, re.split(r"[,\s]+", line.strip()))
                edges.append((src, dst))
            except:
                continue
    print(f"Loaded {len(edges)} edges")
    return edges

def load_and_preprocess(filepath, as_undirected=False):
    # load edges
    edges = load_edgelist(filepath)
    # build ig.Graph
    g = ig.Graph(edges=edges, directed=True)
    
    if as_undirected:
        g = g.as_undirected()

    # extract LCC
    components = g.connected_components()
    giant = components.giant()
    print("Extracted LCC:")
    print(giant.summary())

    return giant

### Greek Wikipedia

In [4]:
filepath = r"..\data\wikipedia_link_el.edges"
lcc = load_and_preprocess(filepath) # load from file and extract lcc

Loaded 5890410 edges
Extracted LCC:
IGRAPH D--- 165633 5628376 -- 


#### Infomap

In [16]:
# custom infomap implementation
comms_custom = opt.search_community_partition(lcc, verbose=False, num_restarts=5)

MemoryError: Error at src/core/vector.c:145: Cannot initialize vector. -- Out of memory

In [ ]:
# igraph infomap implementation 
comms_ig = lcc.community_infomap() # takes ~18min

In [ ]:
# infomap package implementation
results_im = infomap.run(lcc, two_level=True, directed=True, num_trials=5) # takes ~10min

In [ ]:
print("Communities in giant component:", len(comms_ig))
print("Codelength:", comms_ig.codelength)

In [ ]:
compare_partitions(comms_custom, comms_ig.membership)
print("custom description length: ", meq.compute_description_length(lcc, comms_custom))
print("igraph description length: ", comms_ig.codelength)
print("igraph description length w custom computation: ", meq.compute_description_length(lcc, comms_ig.membership))

#### Neuromap

In [5]:
def sparse_from_igraph(G: ig.Graph) -> Tuple[torch.Tensor, Dict[int, int]]:
    """
    Converts an igraph.Graph to a sparse tensor.

    Parameters
    ----------
    G : ig.Graph
        The igraph graph, which can be weighted and/or directed.

    Returns
    -------
    Tuple[torch.Tensor, Dict[int, int]]
        A tuple containing the sparse tensor representation of the input graph
        and a dictionary from zero-based IDs to the original node names.
    """

    has_names = "name" in G.vs.attributes()
    has_weight = "weight" in G.es.attributes()

    # always make sure to sort the nodes so they're in the expected order
    # (sort by "name" attribute if present, otherwise by vertex index)
    the_nodes = sorted(
        range(G.vcount()),
        key=lambda i: G.vs[i]["name"] if has_names else i
    )
    node_to_ID = {node: ID for (ID, node) in enumerate(the_nodes)}
    ID_to_node = {
        ID: (G.vs[node]["name"] if has_names else node)
        for (ID, node) in enumerate(the_nodes)
    }

    # for a directed graph, "neighbors" should mean out-neighbors, to mirror
    # the successors-style adjacency you'd get iterating G.neighbors(u) on a
    # networkx DiGraph; for undirected graphs, all neighbors are used
    mode = "out" if G.is_directed() else "all"

    indices = [[], []]
    values = []
    for u in the_nodes:
        for v in sorted(G.neighbors(u, mode=mode)):  # again, always sorting...
            weight = 1.0
            if has_weight:
                eid = G.get_eid(u, v, directed=G.is_directed())
                w = G.es[eid]["weight"]
                if w is not None:
                    weight = w
            indices[0].append(node_to_ID[u])
            indices[1].append(node_to_ID[v])
            values.append(float(weight))

    return (
        torch.sparse_coo_tensor(
            indices=indices,
            values=values,
            size=(len(the_nodes), len(the_nodes))
        ),
        ID_to_node
    )



def to_dataset(G: Union[nx.Graph, ig.Graph], y_true : List[int]) -> Data:
    """
    Takes a networkx or igraph graph and a list of community labels for the nodes and
    returns them as a pyg Data representation.

    Parameters
    ----------
    G : nx.Graph/igraph.Graoh
        The networkx or igraph graph.

    y_true : List[int]
        List of the nodes' community labels.

    Returns
    -------
    Data
        A Data object where the edge index and node features X are a sparse
        tensor representation of the graph's adjacency matrix and the node
        labels a the nodes' true communities.
    """
    
    if isinstance(G, nx.Graph):
        adj = nm.sparse_from_networkx(G)[0]
    elif isinstance(G, ig.Graph):
        adj = sparse_from_igraph(G)[0]
    else:
        raise TypeError(
            f"G must be a networkx.Graph or igraph.Graph, got {type(G)}"
        )

    data = Data()
    adj = adj.coalesce()

    data.edge_index = adj
    data.x          = adj
    data.y          = torch.Tensor(y_true).long()

    return data

In [ ]:
data = to_dataset(G = lcc, y_true = []) 
n = lcc.vcount()

model = GCN( in_channels     = n
           , hidden_channels = 100  # neuromap paper uses 4*sqrt(n)
           , num_layers      = 2
           , out_channels    = int(np.sqrt(n)) # as in neuromap paper
           , act             = "selu"
           , norm            = "batch"
           , dropout         = 0.5
           )

neuromap = nm.Neuromap(model = model, device = "cpu")
print(neuromap)

Neuromap(
  (model): GCN(216228, 216228, num_layers=2)
)


In [7]:
# setting a high learning rate for this example
# sometimes, the GNN takes a "wrong turn", so in general, it's a good idea to run several trials
L, S = neuromap.fit(data, epochs = 1000, patience = 100, lr = 1e-1)

RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 187018191936 bytes.